# Customer Segmentation

In [58]:
#Import packages
import pandas as pd
import numpy as np
import scipy   #It is a tool used to solve mathematical, statistical, and scientific problems.
import matplotlib.pyplot as plt
import seaborn as sns

In [59]:
#Extract dataset
df = pd.read_csv('https://raw.githubusercontent.com/HumayDS/Digital-Data-Analytics-E-15-24/refs/heads/main/dataset%20rfm.csv')

In [60]:
df.head()

,CustomerID,PurchaseDate,TransactionAmount,ProductInformation,OrderID,Location
0,8814,2023-04-11,943.31,Product C,890075,Tokyo
1,2188,2023-04-11,463.70,Product A,176819,London
2,4608,2023-04-11,80.28,Product A,340062,New York
3,2559,2023-04-11,221.29,Product A,239145,London
4,9482,2023-04-11,739.56,Product A,194545,Paris


In [61]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   CustomerID          1000 non-null   int64  
 1   PurchaseDate        1000 non-null   object 
 2   TransactionAmount   1000 non-null   float64
 3   ProductInformation  1000 non-null   object 
 4   OrderID             1000 non-null   int64  
 5   Location            1000 non-null   object 
dtypes: float64(1), int64(2), object(3)
memory usage: 47.0+ KB


In [62]:
# Change PDate to date format
df['PurchaseDate'] = pd.to_datetime(df['PurchaseDate'])

In [63]:
  #NA's check
df.isna().sum()

,0
CustomerID,0
PurchaseDate,0
TransactionAmount,0
ProductInformation,0
OrderID,0
Location,0


In [64]:
#num of unique customers
df['CustomerID'].nunique()

946

In [65]:
df['ProductInformation'] = df['ProductInformation'].astype('category')
df['Location'] = df['Location'].astype('category')

#RFM  - Recency -- Frequency --  Monetary

#Recency  — shows how many days have passed since the customer’s last purchase until today.

In [66]:
# analysis_date = the date that is 1 day after the last purchase date in the dataset
# If we do not add 1 day:
# The customer who made a purchase on the last day will have a Recency of 0
analysis_date = df['PurchaseDate'].max() + pd.Timedelta(days=1)

In [67]:
analysis_date

Timestamp('2023-06-11 00:00:00')

In [68]:
#There may be repetition here because the same customer can make purchases on different dates.
recency = df.groupby('CustomerID')['PurchaseDate'].max().reset_index()

In [69]:
recency

,CustomerID,PurchaseDate
0,1011,2023-05-08
1,1025,2023-05-20
2,1029,2023-06-10
3,1046,2023-04-28
4,1049,2023-05-28
...,...,...
941,9941,2023-04-29
942,9950,2023-05-03
943,9954,2023-05-29
944,9985,2023-04-14


In [70]:
#How many days have passed since the last purchase?
recency['Recency'] = (analysis_date - recency['PurchaseDate']).dt.days

In [71]:
#A higher recency value means that a long time has passed since the last purchase.
recency

,CustomerID,PurchaseDate,Recency
0,1011,2023-05-08,34
1,1025,2023-05-20,22
2,1029,2023-06-10,1
3,1046,2023-04-28,44
4,1049,2023-05-28,14
...,...,...,...
941,9941,2023-04-29,43
942,9950,2023-05-03,39
943,9954,2023-05-29,13
944,9985,2023-04-14,58


In [72]:
#Eliminate PurchaseDate
recency = recency[['CustomerID', 'Recency']]

In [73]:
recency

,CustomerID,Recency
0,1011,34
1,1025,22
2,1029,1
3,1046,44
4,1049,14
...,...,...
941,9941,43
942,9950,39
943,9954,13
944,9985,58


# Frequency — It shows how many times the customer made purchases within a certain period.

In [74]:
frequency = df.groupby('CustomerID')['OrderID'].count().reset_index()

In [75]:
frequency

,CustomerID,OrderID
0,1011,2
1,1025,1
2,1029,1
3,1046,1
4,1049,1
...,...,...
941,9941,1
942,9950,1
943,9954,1
944,9985,1


In [76]:
#Rename edk
frequency = frequency.rename(columns = {'OrderID': 'Frequency'})

In [77]:
frequency

,CustomerID,Frequency
0,1011,2
1,1025,1
2,1029,1
3,1046,1
4,1049,1
...,...,...
941,9941,1
942,9950,1
943,9954,1
944,9985,1


In [78]:
#Most customers have made a purchase only once.
frequency['Frequency'].value_counts()

,count
Frequency,
1,895
2,48
3,3


#Monetary - Each customer’s total spend.

In [79]:
# Here we will find how much money each customer has spent.
monetary = df.groupby('CustomerID')['TransactionAmount'].sum().reset_index()

monetary.columns = ['CustomerID', 'Monetary']

In [80]:
monetary

,CustomerID,Monetary
0,1011,1129.02
1,1025,359.29
2,1029,704.99
3,1046,859.82
4,1049,225.72
...,...,...
941,9941,960.53
942,9950,679.11
943,9954,798.01
944,9985,36.10


# RFM — In this step, we combine the recency, frequency, and monetary metrics for each customer into a single DataFrame.

This allows us to obtain a complete and overall view of the customer’s behavior.

In [81]:
rfm = recency.merge(frequency, on='CustomerID')
rfm = rfm.merge(monetary, on='CustomerID')

In [82]:
rfm

,CustomerID,Recency,Frequency,Monetary
0,1011,34,2,1129.02
1,1025,22,1,359.29
2,1029,1,1,704.99
3,1046,44,1,859.82
4,1049,14,1,225.72
...,...,...,...,...
941,9941,43,1,960.53
942,9950,39,1,679.11
943,9954,13,1,798.01
944,9985,58,1,36.10


In [83]:
rfm.describe().T

,count,mean,std,min,25%,50%,75%,max
CustomerID,946.0,5566.216702,2598.266606,1011.00,3281.75,5557.500,7824.750,9991.00
Recency,946.0,30.983087,17.318484,1.00,15.00,32.000,45.000,61.00
Frequency,946.0,1.057082,0.245418,1.00,1.00,1.000,1.000,3.00
Monetary,946.0,542.999799,324.382398,12.13,266.64,542.895,782.695,2379.45


## Assign scores (1–5) for R, F, and M
# The lower the Recency, the better (reverse scoring).

For the Recency metric, a lower value means that the customer made their last purchase recently, which helps identify customers who are still active.

On the other hand, a higher Frequency and a higher Monetary value indicate more loyal and higher-value customers.

In [84]:
#This function (pd.qcut) divides the Recency values into 5 equal groups (quantiles).
rfm['R_score'] = pd.qcut(
    rfm['Recency'],
    5,
    labels=[5,4,3,2,1]
)

In [85]:
rfm['M_score'] = pd.qcut(
    rfm['Monetary'],
    5,
    labels=[1,2,3,4,5]
)

In [86]:
#Customers who make 2–3 purchases → active and good customers. →  5 score
#Other customers → normal level. →  4 score
#We cannot assign a bad score to customers who made only one purchase, because they make up a very large portion of our customers. In fact, the company earns from them.
rfm['F_score'] = rfm['Frequency'].apply(
    lambda x: 5 if x in [2, 3] else 4
)

In [87]:
rfm

,CustomerID,Recency,Frequency,Monetary,R_score,M_score,F_score
0,1011,34,2,1129.02,3,5,5
1,1025,22,1,359.29,4,2,4
2,1029,1,1,704.99,5,4,4
3,1046,44,1,859.82,2,5,4
4,1049,14,1,225.72,4,2,4
...,...,...,...,...,...,...,...
941,9941,43,1,960.53,2,5,4
942,9950,39,1,679.11,2,4,4
943,9954,13,1,798.01,5,4,4
944,9985,58,1,36.10,1,1,4


In [88]:
#RFM Calculate

In [89]:
rfm['RFM_segment'] = (
    rfm['R_score'].astype(str) +
    rfm['F_score'].astype(str) +
    rfm['M_score'].astype(str)
)

In [90]:
rfm

,CustomerID,Recency,Frequency,Monetary,R_score,M_score,F_score,RFM_segment
0,1011,34,2,1129.02,3,5,5,355
1,1025,22,1,359.29,4,2,4,442
2,1029,1,1,704.99,5,4,4,544
3,1046,44,1,859.82,2,5,4,245
4,1049,14,1,225.72,4,2,4,442
...,...,...,...,...,...,...,...,...
941,9941,43,1,960.53,2,5,4,245
942,9950,39,1,679.11,2,4,4,244
943,9954,13,1,798.01,5,4,4,544
944,9985,58,1,36.10,1,1,4,141


R_score: 1–5

M_score: 1–5

F_score: 4 and  5

Logic
 VIP Customers

R ≥ 4

M ≥ 4

F = 5

 High Value

R ≥ 3

M ≥ 4

 Medium Value

R ≥ 3

M ≤ 3

 Low Value
All the remaining ones (all customers with R_score ≤ 2).

In [91]:
rfm['Customer_Group'] = 'Low Value'

# Medium
rfm.loc[
    (rfm['R_score'] >= 3) &
    (rfm['M_score'] <= 3),
    'Customer_Group'
] = 'Medium Value'

# High
rfm.loc[
    (rfm['R_score'] >= 3) &
    (rfm['M_score'] >= 4),
    'Customer_Group'
] = 'High Value'

# VIP
rfm.loc[
    (rfm['R_score'] >= 4) &
    (rfm['M_score'] >= 4) &
    (rfm['F_score'] == 5),
    'Customer_Group'
] = 'VIP Customers'

In [92]:
rfm

,CustomerID,Recency,Frequency,Monetary,R_score,M_score,F_score,RFM_segment,Customer_Group
0,1011,34,2,1129.02,3,5,5,355,VIP Customers
1,1025,22,1,359.29,4,2,4,442,Low Value
2,1029,1,1,704.99,5,4,4,544,Low Value
3,1046,44,1,859.82,2,5,4,245,High Value
4,1049,14,1,225.72,4,2,4,442,Low Value
...,...,...,...,...,...,...,...,...,...
941,9941,43,1,960.53,2,5,4,245,High Value
942,9950,39,1,679.11,2,4,4,244,High Value
943,9954,13,1,798.01,5,4,4,544,Low Value
944,9985,58,1,36.10,1,1,4,141,Medium Value


In [93]:
rfm.to_excel("RFM_Table.xlsx", index=False)

In [94]:
rfm['Customer_Group'].value_counts()

,count
Customer_Group,
Low Value,379
Medium Value,339
High Value,200
VIP Customers,28


VIP → the most active and highest-spending customers

High → a strong customer base

Medium → has potential

Low → either inactive (lost) or low-spending customers

In [95]:
segment_revenue = (
    rfm.groupby('Customer_Group')
       .agg(
           Customer_Count=('CustomerID', 'count'),
           Total_Revenue=('Monetary', 'sum')
       )
       .reset_index()
)

# Revenue perc
segment_revenue['Revenue_%'] = (
    segment_revenue['Total_Revenue'] /
    segment_revenue['Total_Revenue'].sum() * 100
).round(2)

# Customer perc
segment_revenue['Customer_%'] = (
    segment_revenue['Customer_Count'] /
    segment_revenue['Customer_Count'].sum() * 100
).round(2)

segment_revenue.sort_values(by='Revenue_%', ascending=False)

,Customer_Group,Customer_Count,Total_Revenue,Revenue_%,Customer_%
1,Low Value,379,201462.10,39.22,40.06
0,High Value,200,163514.13,31.83,21.14
2,Medium Value,339,114932.15,22.37,35.84
3,VIP Customers,28,33769.43,6.57,2.96


#Insight

# 1) The Pareto rule does not work.

Normally:

20% of customers → 80% of revenue.

In our case:

24% (High + VIP) → only 38% of revenue.

This shows that:

Revenue is distributed more evenly among customers, and there is no strong premium concentration.

# 2)Low Value is a warning signal.

40% of customers

39% of revenue

This means:

Even low-value customers generate a considerable amount of revenue.

This may indicate two things:

The segmentation boundaries are too soft .

The customer base has a generally homogeneous structure.

# 3) The VIP segment is very weak.

3% of customers

6.5% of revenue

This is a low indicator for a premium business.

In a strong retail business, normally:

5–10% of customers
generate 20–40% of revenue.

In our case, the premium layer has not formed.

This business:

✔ is stable
✔ has not become premium
✔ does not have a strong VIP layer
✔ has a weak retention system

The main problem:

There is no mechanism that increases customer value (upsell, loyalty, subscription, etc.).

#Final comment
The RFM analysis is technically correct and the calculations appear accurate. However, the results show weak differentiation between segments and a weak VIP (premium) layer. Revenue is not highly concentrated.

This may indicate that:

The current scoring and quantile approach does not reflect real business behavior.

Segment boundaries are not calibrated according to business goals.

Key observations:

Frequency does not create strong differentiation.

The VIP segment generates a very small share of revenue.

The Pareto effect is not observed.

Conclusion

The RFM model should be recalibrated and the analysis repeated considering business needs, customer behavior, and revenue concentration.

Recommended improvements:

Use business-based thresholds instead of quantiles

Create better frequency differentiation

This will allow more accurate segmentation and better strategic decisions.

Proposal – Opinion 2

Current structure:
A stable retail model.

Desired goal:
If the objective is to build a premium and higher-margin business, the segmentation should be tightened and an active upsell strategy should be implemented.